<a href="https://colab.research.google.com/github/EduardoAve/Labour-well-being/blob/main/notebooks/data_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploratory Data Analysis (EDA): Academic Labour Well-being (Austria & Czech Rep.)

This notebook performs a detailed exploratory analysis of the labour well-being dataset for academic staff in Austria and the Czech Republic. The objectives are:
1.  **Clean and preprocess** the data (handling missing values with mode imputation, encoding categorical labels).
2.  Perform **univariate analysis** to understand the distribution of each variable.
3.  Perform **bivariate and multivariate analysis** to explore relationships between variables (correlations, scatter plots, comparative plots).
4.  Investigate **disparities and biases** through grouped statistics and comparative visualizations by country, gender, age, institution type, and field of study.

The analysis focuses on exploration and description, preparing the ground for future statistical models.

## 1. Initial Setup: Importing Libraries

In [ ]:
# Data Manipulation
import pandas as pd
import numpy as np

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gaussian_kde

# Preprocessing (Only Mode Imputation needed now)
# from sklearn.impute import KNNImputer # Removed as per request

# Visualization Utilities and Warnings
from IPython.display import display, Markdown
import warnings
warnings.filterwarnings('ignore') # Suppress warnings for cleaner output

# Visualization Settings
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 5) # Standard figure size

## 2. Data Loading

In [ ]:
# Load the dataset from the CSV file
# Make sure the path is correct for your environment
file_path = 'labour_well_being_data.csv' # Adjust this path as needed
try:
    # If using Google Colab and file is in Drive:
    # from google.colab import drive
    # drive.mount('/content/drive')
    # file_path = '/content/drive/MyDrive/path/to/your/labour_well_being_data.csv'
    df_raw = pd.read_csv(file_path)
    print("Data loaded successfully.")
    display(df_raw.head())
except FileNotFoundError:
    print(f"Error: File '{file_path}' not found. Please check the path.")
    # Create a dummy DataFrame for demonstration if file not found
    print("Creating dummy data for demonstration purposes.")
    data = {
        'Country': np.random.choice([1, 2], 100),
        'Gender': np.random.choice([1, 2, 3], 100),
        'Age': np.random.randint(25, 65, 100),
        'Burnout': np.random.uniform(1, 7, 100),
        'Job_Satisfaction': np.random.uniform(1, 5, 100),
        'Vulnerability': np.random.uniform(1, 4, 100),
        'Marital_Status': np.random.choice([1,2,3,4,5,6, np.nan], 100, p=[0.3,0.2,0.2,0.1,0.1,0.05, 0.05]),
        # Add other columns as needed for the code to run
        'Version': 1,
        'HEI_Type': np.random.choice([1,2,3,4,5,6,7,8,np.nan], 100),
        'Faculty_Subject_Area': np.random.choice(list(range(1,14))+[np.nan], 100),
        'Employment_Contract_Duration': np.random.choice(list(range(1,8))+[np.nan], 100),
        'Leadership_Position': np.random.choice([1,2,3,4,np.nan], 100),
        'Other_Paid_Job': np.random.choice(list(range(1,8))+[np.nan], 100),
        'Academic_or_Non_Academic': np.random.choice([1,2], 100),
        'Current_Position': np.random.choice(list(range(1,10))+[np.nan], 100),
        'Job_Category': np.random.choice(list(range(1,15))+[np.nan], 100),
        'Highest_Education_Level': np.random.choice(list(range(1,9))+[np.nan], 100),
        'Salary_per_Hour': np.random.uniform(10, 50, 100),
        'HEI_Actual_Weekly_Hours': np.random.uniform(20, 60, 100),
        'Perceived_Autonomy': np.random.uniform(1, 5, 100),
        'Performance_Pressure': np.random.uniform(1, 5, 100),
        'Quality_of_Leadership': np.random.uniform(1, 5, 100),
        'Sense_of_Community': np.random.uniform(1, 5, 100)
    }
    df_raw = pd.DataFrame(data)
    # Simulate scaling for CZ in dummy data
    cz_mask_dummy = df_raw['Country'] == 2
    df_raw.loc[cz_mask_dummy, 'Burnout'] = np.random.uniform(1, 5, cz_mask_dummy.sum())
    display(df_raw.head())

    # df_raw = pd.DataFrame() # Fallback to empty if dummy creation fails

## 3. Initial Data Exploration

In [ ]:
if not df_raw.empty:
    print(f"DataFrame Dimensions: {df_raw.shape}")
    print("\nGeneral Information and Data Types:")
    df_raw.info()

    print("\nSummary of Missing Values:")
    missing_values = df_raw.isnull().sum()
    missing_percentage = (missing_values / len(df_raw)) * 100
    missing_info = pd.DataFrame({'Count': missing_values, 'Percentage': missing_percentage})
    display(missing_info[missing_info['Count'] > 0].sort_values(by='Count', ascending=False))

    print("\nInitial Descriptive Statistics (Numerical Variables):")
    # Select only numeric columns for describe(), excluding potential IDs like 'Version'
    numeric_cols_initial = df_raw.select_dtypes(include=np.number).columns.difference(['Version'])
    display(df_raw[numeric_cols_initial].describe().T)
else:
    print("Empty DataFrame, initial exploration skipped.")

## 4. Data Preprocessing

In this section, we will perform the following steps:
1.  **Scale `Burnout`:** Adjust the Czech Republic `Burnout` scale (1-5) to match the Austrian scale (1-7).
2.  **Label Encoding:** Convert numerical categorical variables to readable text labels.
3.  **Missing Value Imputation:** Use the **Mode** (most frequent value) for categorical variables.

### 4.1 Scale `Burnout` Variable

In [ ]:
if not df_raw.empty:
    df = df_raw.copy() # Work on a copy

    # Check if 'Burnout' and 'Country' columns exist
    if 'Burnout' in df.columns and 'Country' in df.columns:
        print("Scaling 'Burnout' for Czech Republic (Country=2) from 1-5 to 1-7 scale...")

        # Identify Czech Republic respondents (Country code 2)
        cz_mask = df['Country'] == 2

        # Store original values before scaling for comparison
        original_burnout_cz = df.loc[cz_mask, 'Burnout'].copy()

        # Apply linear scaling: new = new_min + (old - old_min) * (new_max - new_min) / (old_max - old_min)
        # Here: old_min=1, old_max=5, new_min=1, new_max=7
        df.loc[cz_mask, 'Burnout'] = 1 + (df.loc[cz_mask, 'Burnout'] - 1) * (7 - 1) / (5 - 1)

        print("'Burnout' scaling completed.")
        # Display comparison for a few CZ entries if available
        if cz_mask.any():
             comparison_df = pd.DataFrame({
                 'Country': df.loc[cz_mask, 'Country'],
                 'Original Burnout (1-5)': original_burnout_cz,
                 'Scaled Burnout (1-7)': df.loc[cz_mask, 'Burnout']
             })
             display(comparison_df.head())
        else:
             print("No Czech Republic data found to show scaling comparison.")

    else:
        print("Warning: 'Burnout' or 'Country' column not found. Scaling skipped.")
        # If df wasn't created yet because df_raw was empty
        if 'df' not in locals():
             df = df_raw.copy() # Still need df for next steps
             if df.empty:
                  print("Warning: DataFrame is empty after copy.")
else:
    print("Empty raw DataFrame, Burnout scaling skipped.")
    df = pd.DataFrame() # Ensure df exists

### 4.2 Label Encoding for Categorical Variables

In [ ]:
if not df.empty:

    # --- Define Mappings based on Data Dictionary and Questionnaire (Updated) ---
    country_map = {1: 'Austria', 2: 'Czech Republic'}
    gender_map = {1: 'Male', 2: 'Female', 3: 'Other'}
    marital_map = {
        1: 'Married/Registered Partnership', 2: 'In a relationship (unmarried)',
        3: 'Single', 4: 'Divorced', 5: 'Widowed', 6: 'Other_Marital'
    }
    care_map = {
        1: 'No', 2: 'Care for underage children', 3: 'Care for dependent relatives',
        4: 'Combination Care'
    }
    # Combined HEI Type map for simplicity in visualization, though codes differ by country
    hei_type_map = {
        1: 'CZ: Public HEI', 2: 'CZ: Private HEI', 3: 'CZ: State HEI',
        4: 'AT: Public University', 5: 'AT: Private University/College',
        6: 'AT: University of Applied Sciences',
        7: 'AT: Public Uni College Teacher Ed', 8: 'AT: Private Uni College Teacher Ed'
    }
    faculty_map = {
        1: 'Natural sciences', 2: 'Technical sciences', 3: 'Agricultural/forestry/veterinary',
        4: 'Healthcare/medical/pharmaceutical', 5: 'Humanities/social sciences',
        6: 'Economic sciences', 7: 'Law', 8: 'Pedagogy/teacher training',
        9: 'Culture/art', 10: 'Sport sciences', 11: 'Unspecified/cannot be categorised',
        12: 'Security/defence/Military', 13: 'Other_Faculty'
    }
    contract_map = {
        1: 'Permanent/Continuous', 2: 'Fixed-term (permanent prospects)',
        3: 'Fixed-term (no permanent prospects)', 4: 'Casual/hourly',
        5: 'Fixed-term (unspecified prospects)', 6: 'Permanent (tenured AT)',
        7: 'Other_Contract'
    }
    leadership_map = {
        1: 'No', 2: 'Yes (Institution/Faculty/Dept)', 3: 'Yes (Research Team)',
        4: 'Combination Leadership'
    }
    # Policy_Influence is ordinal, keep numeric for now
    other_job_map = {
        1: 'No', 2: 'Yes (Public Sector)', 3: 'Yes (Private Sector)',
        4: 'Yes (Non-profit)', 5: 'Yes (Self-employed)',
        6: 'Yes (Multiple Areas)', 7: 'Yes (Other_Job)'
    }
    academic_map = {1: 'Non-academic', 2: 'Academic'}

    # Simplified Position map - combining CZ/AT might be complex, using CZ primary for now
    # Consider creating separate AT/CZ position variables if needed later
    position_map = { # Mapping for Current_Position (Q15 - CZ focus)
        1: 'Non-academic', # From PDF Q15 CZ
        2: 'Lecturer (CZ)', 3: 'Assistant (CZ)', 4: 'Assistant professor (CZ)',
        5: 'Docent (CZ)', 6: 'Professor (CZ)', 7: 'Researcher (CZ)',
        8: 'Externist (CZ)', 9: 'Researcher & Academic (CZ)',
        # AT codes 2-13 are distinct roles, mapping them here might be confusing
        # Add a placeholder if AT codes appear in this column unexpectedly
        10: 'AT Position (See PDF)', 11: 'AT Position (See PDF)',
        12: 'AT Position (See PDF)', 13: 'AT Position (See PDF)'
    }
    job_cat_map = { # Mapping for Job_Category (Q16, Non-academic)
        1: 'Dept/manager assistant', 2: 'Lab technician', 3: 'Librarian/archivist',
        4: 'Facility management', 5: 'ICT', 6: 'Student support',
        7: 'Economics/finance/HR', 8: 'Project management', 9: 'Legal/control',
        10: 'Marketing/PR', 11: 'Science/knowledge transfer', 12: 'Foreign affairs',
        13: 'Other administrative', 14: 'Other/Combination Admin'
    }
    education_map = { # Mapping for Highest_Education_Level (Q17)
        1: 'Elementary', 2: 'Apprenticeship', 3: 'Vocational/Commercial School',
        4: 'High school/Secondary', 5: 'Higher professional school (CZ)',
        6: 'Bachelor', 7: 'Master', 8: 'Doctoral/PhD'
    }

    # --- Apply Mappings ---
    # Create new columns for labels to preserve original numeric codes if needed
    df['Country_Label'] = df['Country'].map(country_map).astype('category')
    df['Gender_Label'] = df['Gender'].map(gender_map).astype('category')
    df['Marital_Status_Label'] = df['Marital_Status'].map(marital_map).astype('category')
    df['Care_Responsibilities_Label'] = df['Care_Responsibilities'].map(care_map).astype('category')
    df['HEI_Type_Label'] = df['HEI_Type'].map(hei_type_map).astype('category')
    df['Faculty_Subject_Area_Label'] = df['Faculty_Subject_Area'].map(faculty_map).astype('category')
    df['Employment_Contract_Duration_Label'] = df['Employment_Contract_Duration'].map(contract_map).astype('category')
    df['Leadership_Position_Label'] = df['Leadership_Position'].map(leadership_map).astype('category')
    df['Other_Paid_Job_Label'] = df['Other_Paid_Job'].map(other_job_map).astype('category')
    df['Academic_or_Non_Academic_Label'] = df['Academic_or_Non_Academic'].map(academic_map).astype('category')

    # Mappings for subgroup-specific variables (create new labeled columns)
    df['Current_Position_Label'] = df['Current_Position'].map(position_map).astype('category')
    df['Job_Category_Label'] = df['Job_Category'].map(job_cat_map).astype('category')
    df['Highest_Education_Label'] = df['Highest_Education_Level'].map(education_map).astype('category')

    # Select conceptually categorical columns (now with '_Label' suffix)
    categorical_cols_labeled = [col for col in df.columns if col.endswith('_Label')]

    print("Categorical variables encoded with labels:")
    display(df[categorical_cols_labeled].head())
    print("\nData types after label encoding:")
    df.info(verbose=True) # Verify Dtype changes, show all columns
else:
    print("Empty DataFrame, label encoding skipped.")

### 4.3 Missing Value Imputation (Mode for Categorical)

* **Categorical Variables (with labels):** Use the **Mode** (most frequent value).

In [ ]:
if not df.empty:
    # Identify categorical columns (labeled ones)
    categorical_cols_for_imputation = [col for col in df.columns if col.endswith('_Label')]

    # --- Categorical Imputation with Mode ---
    print(f"\nImputing {len(categorical_cols_for_imputation)} labeled categorical columns with Mode...")
    imputed_categorical_count = 0
    for col in categorical_cols_for_imputation:
        if df[col].isnull().any():
            # Check if mode exists (important for columns with all NaNs initially)
            if not df[col].mode().empty:
                mode_val = df[col].mode()[0]
                df[col].fillna(mode_val, inplace=True)
                imputed_categorical_count += 1

                # Ensure dtype remains category after fillna
                # It's crucial to preserve the categories known before imputation
                original_categories = df[col].dtype.categories
                if not pd.api.types.is_categorical_dtype(df[col]):
                    try:
                        # Attempt explicit conversion back to category with original categories
                        df[col] = pd.Categorical(df[col], categories=original_categories, ordered=False)
                    except Exception as e:
                        print(f"Warning: Could not convert column {col} back to category after imputation: {e}")
            else:
                 print(f"Warning: Could not impute column {col} with mode (possibly all values are NaN or no mode exists).")

    print(f"Categorical imputation completed for {imputed_categorical_count} columns.")

    # --- Impute Numeric Columns (Example: using median, as KNN was removed) ---
    # Identify numeric columns excluding IDs and original categoricals
    numeric_cols_final = df.select_dtypes(include=np.number).columns
    cols_to_exclude = ['Version', 'Country', 'Gender', 'Marital_Status', 'Care_Responsibilities',
                       'HEI_Type', 'Faculty_Subject_Area', 'Employment_Contract_Duration',
                       'Leadership_Position', 'Other_Paid_Job', 'Academic_or_Non_Academic',
                       'Current_Position', 'Job_Category', 'Highest_Education_Level']
    numeric_cols_to_impute = numeric_cols_final.difference(cols_to_exclude, sort=False)

    print(f"\nImputing {len(numeric_cols_to_impute)} numeric columns with Median...")
    imputed_numeric_count = 0
    for col in numeric_cols_to_impute:
        if df[col].isnull().any():
            median_val = df[col].median()
            df[col].fillna(median_val, inplace=True)
            imputed_numeric_count += 1
    print(f"Numeric imputation completed for {imputed_numeric_count} columns.")

    # --- Final Check for Missing Values ---
    print("\nMissing values after all imputation:")
    missing_after = df.isnull().sum()
    display(missing_after[missing_after > 0])

    # Assign the processed DataFrame to df_cleaned for consistency
    df_cleaned = df.copy()
    print("\nDataFrame after preprocessing (labeling, scaling, mode/median imputation):")
    display(df_cleaned.head())
    print("\nFinal data types check:")
    df_cleaned.info(verbose=True)

else:
    print("Empty DataFrame, imputation skipped.")
    df_cleaned = pd.DataFrame() # Ensure df_cleaned exists

## 5. Exploratory Data Analysis (EDA)

Now we will explore the cleaned dataset (`df_cleaned`).

### 5.1 Univariate Analysis

In [ ]:
if not df_cleaned.empty:
    # --- Numerical Descriptive Statistics ---
    display(Markdown("#### Descriptive Statistics (Numerical Variables)"))
    # Select only numeric columns, exclude IDs like 'Version' and original categorical codes if desired
    numeric_cols_final = df_cleaned.select_dtypes(include=np.number).columns
    # Optionally exclude original coded categoricals if only labels are needed now
    cols_to_exclude_stats = ['Version', 'Country', 'Gender', 'Marital_Status', 'Care_Responsibilities',
                             'HEI_Type', 'Faculty_Subject_Area', 'Employment_Contract_Duration',
                             'Leadership_Position', 'Other_Paid_Job', 'Academic_or_Non_Academic',
                             'Current_Position', 'Job_Category', 'Highest_Education_Level']
    numeric_cols_for_stats = numeric_cols_final.difference(cols_to_exclude_stats, sort=False)
    if not numeric_cols_for_stats.empty:
        display(df_cleaned[numeric_cols_for_stats].describe().T)
    else:
        print("No numerical columns found for descriptive statistics after exclusions.")

    # --- Categorical Frequencies ---
    display(Markdown("#### Frequencies (Categorical Variables)"))
    # Use the labeled columns created earlier
    categorical_cols_final = [col for col in df_cleaned.columns if col.endswith('_Label')]
    for col in categorical_cols_final:
        display(Markdown(f"##### {col}"))
        # Calculate frequency and percentage
        freq_table = df_cleaned[col].value_counts().to_frame(name="Frequency")
        freq_table['Percentage'] = (df_cleaned[col].value_counts(normalize=True) * 100).round(2)
        display(freq_table)
else:
    print("Empty DataFrame, univariate analysis skipped.")

In [ ]:
if not df_cleaned.empty:
    display(Markdown("#### Univariate Visualizations"))

    # Define consistent palette
    uni_palette = "viridis" # Or 'muted', 'pastel', etc.
    fig_size_uni = (12, 5) # Consistent figure size

    # --- Plots for Numerical Variables (Histogram and Boxplot) ---
    # Use the same selection as for descriptive stats
    numeric_cols_final = df_cleaned.select_dtypes(include=np.number).columns
    cols_to_exclude_plots = ['Version', 'Country', 'Gender', 'Marital_Status', 'Care_Responsibilities',
                             'HEI_Type', 'Faculty_Subject_Area', 'Employment_Contract_Duration',
                             'Leadership_Position', 'Other_Paid_Job', 'Academic_or_Non_Academic',
                             'Current_Position', 'Job_Category', 'Highest_Education_Level']
    numeric_cols_for_plots = numeric_cols_final.difference(cols_to_exclude_plots, sort=False)

    if not numeric_cols_for_plots.empty:
        display(Markdown("##### Numerical Distributions"))
        for col in numeric_cols_for_plots:
            try:
                fig, axes = plt.subplots(1, 2, figsize=(fig_size_uni[0]*1.2, fig_size_uni[1]*0.8)) # Slightly wider for 2 plots

                # Histogram with KDE
                sns.histplot(df_cleaned[col], kde=True, ax=axes[0], bins=30, color=sns.color_palette(uni_palette, 1)[0])
                axes[0].set_title(f'Histogram of {col}')
                axes[0].set_xlabel(col)
                axes[0].set_ylabel('Frequency')

                # Boxplot
                sns.boxplot(x=df_cleaned[col], ax=axes[1], color=sns.color_palette(uni_palette, 3)[1]) # Use a different color from palette
                axes[1].set_title(f'Boxplot of {col}')
                axes[1].set_xlabel(col)

                plt.tight_layout()
                plt.show()
            except Exception as e:
                 print(f"Could not plot numerical variable {col}. Error: {e}")
                 if plt.gcf().get_axes(): plt.close() # Close figure if error occurred
    else:
        print("No numerical columns found for plotting after exclusions.")

    # --- Plots for Categorical Variables (Bar Chart) ---
    # Use the labeled columns
    categorical_cols_final = [col for col in df_cleaned.columns if col.endswith('_Label')]
    display(Markdown("##### Categorical Distributions"))
    for col in categorical_cols_final:
        # Adjust figure height based on number of categories
        num_categories = df_cleaned[col].nunique()
        dynamic_height = max(5, num_categories * 0.4) # Adjust multiplier as needed
        plt.figure(figsize=(fig_size_uni[0]*0.8, dynamic_height))

        # Order bars by frequency descending for better visualization
        try:
             order = df_cleaned[col].value_counts().index
             sns.countplot(y=df_cleaned[col], order=order, palette=uni_palette) # Use 'y' for potentially long labels
             plt.title(f'Distribution of {col.replace("_Label","")}')
             plt.xlabel('Frequency')
             plt.ylabel('') # Hide y-label as it's clear from title/ticks
             plt.tight_layout()
             plt.show()
        except Exception as e:
             print(f"Could not plot categorical variable {col}. Error: {e}")
             if plt.gcf().get_axes(): plt.close() # Close the potentially empty figure

else:
    print("Empty DataFrame, univariate visualizations skipped.")

### 5.2 Bivariate and Multivariate Analysis

Explore relationships between pairs of variables and among multiple variables using scatter plots, correlation matrices, and other comparative plots.

In [ ]:
if not df_cleaned.empty:
    display(Markdown("#### Scatter Plot Matrix (Pairplot) for Key Numerical Variables"))

    # Select a subset of key numerical variables for the pairplot
    # Added Vulnerability
    pairplot_vars = [
        'Age', 'Job_Satisfaction', 'Burnout', 'Salary_per_Hour',
        'HEI_Actual_Weekly_Hours', 'Perceived_Autonomy', 'Performance_Pressure',
        'Vulnerability'
    ]
    # Filter to only include variables actually present in the cleaned dataframe
    pairplot_vars_present = [var for var in pairplot_vars if var in df_cleaned.columns and pd.api.types.is_numeric_dtype(df_cleaned[var])]

    if len(pairplot_vars_present) > 1 and 'Country_Label' in df_cleaned.columns:
        print(f"Generating pairplot for: {pairplot_vars_present}")
        # Use hue='Country_Label' to see differences between countries
        try:
            sns.pairplot(df_cleaned[pairplot_vars_present + ['Country_Label']].dropna(subset=pairplot_vars_present), # Drop rows with NaN in numeric vars only for pairplot
                         hue='Country_Label',
                         diag_kind='kde', # Kernel density estimate on diagonals
                         plot_kws={'alpha': 0.6, 's': 80, 'edgecolor': 'k'}, # Scatter plot settings
                         height=2.5) # Size of each subplot
            plt.suptitle('Scatter Plot Matrix of Key Numerical Variables by Country', y=1.02, fontsize=16)
            plt.show()
        except Exception as e:
            print(f"Could not generate pairplot. Error: {e}")
            if plt.gcf().get_axes(): plt.close()
    elif len(pairplot_vars_present) <= 1:
        print("Not enough numerical variables found for pairplot.")
    else:
        print("Country_Label column missing, cannot generate pairplot with hue.")

else:
    print("Empty DataFrame or missing key variables, pairplot skipped.")

In [ ]:
if not df_cleaned.empty:
    display(Markdown("#### Correlation Matrix (Quantitative Variables)"))

    # Select only quantitative (numerical) columns for correlation
    # Reuse the selection from univariate stats
    numeric_cols_final = df_cleaned.select_dtypes(include=np.number).columns
    cols_to_exclude_corr = ['Version', 'Country', 'Gender', 'Marital_Status', 'Care_Responsibilities',
                           'HEI_Type', 'Faculty_Subject_Area', 'Employment_Contract_Duration',
                           'Leadership_Position', 'Other_Paid_Job', 'Academic_or_Non_Academic',
                           'Current_Position', 'Job_Category', 'Highest_Education_Level']
    numeric_cols_for_corr = numeric_cols_final.difference(cols_to_exclude_corr, sort=False)

    if not numeric_cols_for_corr.empty:
        # Calculate Pearson correlation matrix
        correlation_matrix = df_cleaned[numeric_cols_for_corr].corr(method='pearson')

        # Create mask for the upper triangle
        mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))

        # Set up and display the heatmap
        plt.figure(figsize=(18, 15))
        sns.heatmap(correlation_matrix,
                    mask=mask,
                    cmap='coolwarm',
                    vmax=1, vmin=-1,
                    center=0,
                    linewidths=.5,
                    cbar_kws={"shrink": .7},
                    annot=True, # Show values on the heatmap
                    fmt=".2f") # Format for annotation values
        plt.title('Pearson Correlation Matrix between Quantitative Variables', fontsize=16)
        plt.xticks(rotation=60, ha='right')
        plt.yticks(rotation=0)
        plt.tight_layout()
        plt.show()
    else:
        print("No quantitative columns found for correlation analysis after exclusions.")
else:
    print("Empty DataFrame, correlation analysis skipped.")

#### Additional Bivariate Plots for Bias Identification

In [ ]:
if not df_cleaned.empty:
    display(Markdown("##### Comparing Key Metrics Across Groups"))

    # Define consistent palette for these plots
    biv_palette = "pastel"
    fig_size_biv = (10, 6) # Consistent size

    # --- Example 1: Salary per Hour vs. Gender ---
    if 'Salary_per_Hour' in df_cleaned.columns and 'Gender_Label' in df_cleaned.columns:
        plt.figure(figsize=fig_size_biv)
        sns.boxplot(x='Gender_Label', y='Salary_per_Hour', data=df_cleaned, palette=biv_palette)
        plt.title('Salary per Hour Distribution by Gender')
        plt.xlabel('Gender')
        plt.ylabel('Salary per Hour (Adjusted Euros)')
        plt.xticks(rotation=0)
        plt.tight_layout()
        plt.show()
    else:
        print("Skipping Salary vs Gender plot (missing columns).")

    # --- Example 2: Job Satisfaction vs. Employment Contract Duration ---
    if 'Job_Satisfaction' in df_cleaned.columns and 'Employment_Contract_Duration_Label' in df_cleaned.columns:
        # Adjust width dynamically for potentially many categories
        num_cat_contract = df_cleaned['Employment_Contract_Duration_Label'].nunique()
        dynamic_width_contract = max(fig_size_biv[0], num_cat_contract * 1.5)
        plt.figure(figsize=(dynamic_width_contract, fig_size_biv[1]))
        # Using violin plot to show density
        sns.violinplot(x='Employment_Contract_Duration_Label', y='Job_Satisfaction', data=df_cleaned, palette=biv_palette, cut=0, inner='quartile')
        plt.title('Job Satisfaction by Employment Contract Duration')
        plt.xlabel('Contract Duration')
        plt.ylabel('Job Satisfaction Score')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
    else:
        print("Skipping Job Satisfaction vs Contract plot (missing columns).")

    # --- Example 3: Burnout vs. Academic/Non-Academic ---
    if 'Burnout' in df_cleaned.columns and 'Academic_or_Non_Academic_Label' in df_cleaned.columns:
        plt.figure(figsize=(fig_size_biv[0]*0.8, fig_size_biv[1])) # Slightly narrower for 2 categories
        sns.boxplot(x='Academic_or_Non_Academic_Label', y='Burnout', data=df_cleaned, palette=biv_palette)
        plt.title('Burnout Score by Staff Type (Academic vs. Non-Academic)')
        plt.xlabel('Staff Type')
        plt.ylabel('Burnout Score (Scaled 1-7)')
        plt.xticks(rotation=0)
        plt.tight_layout()
        plt.show()
    else:
        print("Skipping Burnout vs Staff Type plot (missing columns).")

    # --- Example 4: Vulnerability vs Country ---
    if 'Vulnerability' in df_cleaned.columns and 'Country_Label' in df_cleaned.columns:
        plt.figure(figsize=(fig_size_biv[0]*0.8, fig_size_biv[1]))
        sns.boxplot(x='Country_Label', y='Vulnerability', data=df_cleaned, palette=biv_palette)
        plt.title('Vulnerability Score by Country')
        plt.xlabel('Country')
        plt.ylabel('Vulnerability Score (1-4)')
        plt.xticks(rotation=0)
        plt.tight_layout()
        plt.show()
    else:
        print("Skipping Vulnerability vs Country plot (missing columns).")

    # --- Example 5: Crosstabulation Heatmap - Country vs. HEI Type ---
    if 'Country_Label' in df_cleaned.columns and 'HEI_Type_Label' in df_cleaned.columns:
        display(Markdown("##### Count of HEI Types by Country"))
        try:
            crosstab_hei = pd.crosstab(df_cleaned['Country_Label'], df_cleaned['HEI_Type_Label'])
            plt.figure(figsize=(12, 7))
            sns.heatmap(crosstab_hei, annot=True, fmt='d', cmap='YlGnBu', linewidths=.5)
            plt.title('Frequency of HEI Types within Each Country')
            plt.xlabel('HEI Type')
            plt.ylabel('Country')
            plt.yticks(rotation=0)
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print(f"Could not generate Country vs HEI Type heatmap. Error: {e}")
            if plt.gcf().get_axes(): plt.close()
    else:
        print("Skipping Country vs HEI Type heatmap (missing columns).")

else:
    print("Empty DataFrame, additional bivariate plots skipped.")

### 5.3 Disparity Analysis (Grouped Statistics and Comparative Plots)

We will explore how key well-being variables vary according to:
* `Country_Label`
* `Gender_Label`
* `HEI_Type_Label`
* `Faculty_Subject_Area_Label`
* Age Group (`Age_Group` - to be created)

In [ ]:
if not df_cleaned.empty:
    # Create age groups
    age_bins = [0, 29, 39, 49, 59, np.inf] # Define boundaries: <=29, 30-39, 40-49, 50-59, 60+
    age_labels = ['<30', '30-39', '40-49', '50-59', '60+']
    if 'Age' in df_cleaned.columns:
        df_cleaned['Age_Group'] = pd.cut(df_cleaned['Age'], bins=age_bins, labels=age_labels, right=True)
        # Convert Age_Group to category type explicitly
        df_cleaned['Age_Group'] = df_cleaned['Age_Group'].astype('category')
        print("'Age_Group' column created.")
        display(df_cleaned[['Age', 'Age_Group']].head())
    else:
        print("'Age' column not found, could not create 'Age_Group'.")
else:
    print("Empty DataFrame, age group creation skipped.")

In [ ]:
# Check if df_cleaned exists and is not empty, and if Age_Group was created
if 'df_cleaned' in locals() and not df_cleaned.empty and 'Age_Group' in df_cleaned.columns:
    # --- Grouping Variables and Target Variables ---
    # Use the labeled categorical columns + Age_Group
    grouping_vars_labels = ['Country_Label', 'Gender_Label', 'HEI_Type_Label',
                            'Faculty_Subject_Area_Label', 'Age_Group',
                            'Employment_Contract_Duration_Label', 'Leadership_Position_Label']

    # Select key well-being variables (ensure they exist and are numeric)
    # Added Vulnerability
    target_vars = [
        'Job_Satisfaction', 'Burnout', 'Perceived_Autonomy',
        'Performance_Pressure', 'Quality_of_Leadership', 'Sense_of_Community',
        'Salary_per_Hour', 'HEI_Actual_Weekly_Hours', 'Vulnerability'
    ]

    # Filter target_vars to ensure they exist in df_cleaned and are numeric
    valid_target_vars = [var for var in target_vars if var in df_cleaned.columns and pd.api.types.is_numeric_dtype(df_cleaned[var])]
    print(f"Well-being variables to analyze: {valid_target_vars}")

    # Define consistent palette for grouped plots
    grouped_palette = "muted"

    # --- Generate Grouped Statistics and Plots ---
    for group_var in grouping_vars_labels:
        if group_var in df_cleaned.columns:
            # Check if the grouping variable has non-missing values
            if df_cleaned[group_var].notna().any():
                display(Markdown(f"#### Grouped Analysis by: {group_var}"))

                # Calculate grouped descriptive statistics (mean and median)
                try:
                    # Use observed=False for category dtype if needed (Pandas >= 1.5)
                    # Also handle potential non-numeric grouping vars if any slipped through
                    if pd.api.types.is_categorical_dtype(df_cleaned[group_var]) and pd.__version__ >= '1.5.0':
                         grouped_stats = df_cleaned.groupby(group_var, observed=False)[valid_target_vars].agg(['mean', 'median'])
                    else:
                         grouped_stats = df_cleaned.groupby(group_var)[valid_target_vars].agg(['mean', 'median'])
                    display(grouped_stats)
                except Exception as e:
                     print(f"Could not calculate grouped stats for {group_var}. Error: {e}")
                     continue # Skip plotting if stats failed

                # Generate comparative plots (Boxplots or Violinplots)
                for target_var in valid_target_vars:
                    # Determine appropriate figure size based on number of categories
                    try:
                        num_categories = df_cleaned[group_var].nunique()
                        fig_height = 6
                        fig_width = max(10, num_categories * 1.2) # Adjust width based on categories
                        plt.figure(figsize=(fig_width, fig_height))

                        # Use violinplot to show distribution plus comparison
                        sns.violinplot(x=group_var, y=target_var, data=df_cleaned, palette=grouped_palette, cut=0, inner='quartile',
                                       order=grouped_stats.index) # Ensure order matches stats table
                        # Alternative: sns.boxplot(x=group_var, y=target_var, data=df_cleaned, palette=grouped_palette, order=grouped_stats.index)

                        plt.title(f'{target_var} by {group_var.replace("_Label","")}')
                        plt.xlabel(group_var.replace('_Label', '')) # Cleaner axis label
                        plt.ylabel(target_var)
                        # Rotate X-axis labels if there are many or they are long
                        if num_categories > 5:
                            plt.xticks(rotation=45, ha='right')
                        else:
                            plt.xticks(rotation=0)
                        plt.tight_layout()
                        plt.show()
                    except Exception as e:
                         print(f"Could not plot {target_var} by {group_var}. Error: {e}")
                         if plt.gcf().get_axes(): # Close the potentially empty figure
                             plt.close()
            else:
                print(f"Skipping grouped analysis for '{group_var}' as it contains only missing values or is not present.")
        else:
             print(f"Warning: Grouping variable '{group_var}' not found in the DataFrame.")

else:
    print("Empty DataFrame or 'Age_Group' not created, grouped analysis skipped.")

## 6. Preliminary EDA Conclusions and Next Steps

The exploratory analysis has provided a detailed view of the dataset's characteristics and revealed initial patterns and potential disparities in labour well-being.

**Key Observations:**
* The distributions of numerical and categorical variables, including the newly added `Vulnerability`, have been identified and visualized consistently.
* The correlation analysis shows some linear relationships between numerical variables but also indicates the need for more complex models to capture non-linear relationships.
* Bivariate plots and grouped analysis have begun to highlight differences in well-being perceptions (including Vulnerability) based on country, gender, age, institution type, contract type, and field of study. These differences warrant deeper investigation through statistical tests (e.g., t-tests, ANOVA) and multivariate modeling.
* The scaling of the `Burnout` variable allows for direct comparison between Austria and the Czech Republic on this metric.

**Next Steps:**
1.  **Inferential Statistical Analysis:** Perform hypothesis tests to confirm if the differences observed in the grouped analysis are statistically significant.
2.  **Feature Selection/Engineering:** Based on the EDA and domain knowledge, select the most relevant predictor and outcome variables for modeling. Consider creating interaction terms or composite scores if theoretically justified.
3.  **Modeling:** Develop models (e.g., multiple linear regression, logistic/ordinal regression, structural equation modeling) to identify the key determinants of labour well-being and vulnerability, controlling for multiple factors.
4.  **Encoding for Models:** For modeling, the categorical variables with text labels will need to be numerically encoded (e.g., one-hot encoding, dummy variables).
5.  **Scaling:** Numerical variables may need to be scaled (e.g., standardization) before applying certain modeling algorithms, especially distance-based ones.